In [1]:
import lightgbm
import mlflow
import os
from dotenv import load_dotenv
import sys
import pandas as pd
import json

In [2]:
pd.set_option("display.max_columns", 100)
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
sys.path.append(src_path)
json_path = os.path.join(data_path, "processed/split_info.json")
with open(json_path) as f:
    json_info = json.load(f)
train_end = json_info.get("train_end")
val_end = json_info.get("validation_end")
from about_data.data_load import load_df
from about_data.split import temporal_split
from features.engineering import create_d_features, create_advanced_time_features, add_distance_features, add_interaction_features, add_all_features
from model.preprocessor_pipe_evalueate import create_pipeline, evaluate_model, get_preprocessor

full_df = load_df(data_path)
train, val, test = temporal_split(full_df, train_end, val_end)

In [3]:
map_dfs = {"train": train, "val": val, "test": test}

train_base = create_d_features(train)
val_base = create_d_features(val)

train_features_dfs = {
    "d": train_base,
    "time": create_advanced_time_features(train_base),
    "distance": add_distance_features(train_base),
    "interaction": add_interaction_features(train_base),
    "all": add_all_features(train),
}
val_features_dfs = {
    "d": val_base,
    "time": create_advanced_time_features(val_base),
    "distance": add_distance_features(val_base),
    "interaction": add_interaction_features(val_base),
    "all": add_all_features(val),
}

In [4]:
len(train_features_dfs)

5

In [5]:
y_datasets = {}

for name, sample_df in map_dfs.items():
    y_datasets[name] = sample_df['isFraud']

In [6]:
model = lightgbm.LGBMClassifier(n_estimators=300, learning_rate=0.05, objective="binary", metric='average_precision', random_state=42, n_jobs=-1)

In [7]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud-detection-feature-engineering")

results = []

for name in train_features_dfs:
    with mlflow.start_run(run_name=f"lightgbm_{name}_features"):
        X_train = train_features_dfs[name]
        X_val = val_features_dfs[name]

        pipe = create_pipeline(model, get_preprocessor(X_train))
        pipe.fit(X_train, y_datasets["train"])
        metrics = evaluate_model(pipe, X_val, y_datasets["val"])

        mlflow.log_param("dataset", name)
        mlflow.log_param("feature_count", X_train.shape[1])
        mlflow.log_param("model", "lightgbm")
        mlflow.log_metrics(metrics)

        results.append({"feature_set": name, **metrics})

[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.084459 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11152
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 4097
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794
🏃 View run lightgbm_d_features at: http://127.0.0.1:5000/#/experiments/4/runs/acca12b821fe441bb209bb75db5532f9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.088006 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you

In [11]:
pd.DataFrame(results).sort_values("pr_auc", ascending = False)

,feature_set,pr_auc,roc_auc,precision,recall,f1
4,all,0.375561,0.879628,0.665131,0.142341,0.234498
3,interaction,0.374009,0.880371,0.674491,0.141683,0.234175
0,d,0.372754,0.878931,0.672673,0.147272,0.241640
2,distance,0.372147,0.879338,0.672593,0.149244,0.244283
1,time,0.370914,0.876728,0.675150,0.148258,0.243127
